# RASM-Vul — End-to-End Prototype

This notebook implements an **RASM-Vul-inspired** pipeline:

**PrimeVul paired data → preprocessing → vulnerable/fixed pairing → knowledge → 5 views → embeddings/indexes → multi-view retrieval → WRRF → LLM-style reranking → JSON output**

Important:
- This is a **working research prototype**, not a byte-for-byte reproduction of the paper.
- The paper uses DeepSeek-V2.5 Coder for knowledge extraction, UniXCoder 768-d embeddings, five indices, WRRF, and an LLM for re-ranking/decision.
- The default notebook keeps the expensive LLM steps optional. You can turn them on later.
- Keep validation/test data completely outside the retrieval index.


# Phase 1 run order — important

For the current Phase-1 implementation, run the notebook cells in order.

The knowledge-extraction section now follows this order:

1. Knowledge schema + fallback
2. Knowledge-extraction prompt/parser
3. DeepSeek API connection
4. Test ONE pair
5. Build knowledge records
6. Attach the generated knowledge to `train_pairs`

`KNOWLEDGE_MAX_PAIRS = 1` is intentional. It prevents accidentally sending all training pairs to the API while we are debugging.

After the one-pair test succeeds, change it to `None` when you are ready to process the full training-pair set.


In [2]:
# 1. Install dependencies (run once if needed)
# !pip install pandas numpy scikit-learn tree-sitter tree-sitter-cpp
# !pip install transformers torch sentencepiece
# !pip install faiss-cpu
# Optional LLM/API packages can be installed separately.

import os, re, json, math, hashlib, warnings
import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import classification_report, confusion_matrix, f1_score, recall_score, accuracy_score

warnings.filterwarnings("ignore")
SEED = 42
np.random.seed(SEED)

print("Environment ready")


Environment ready


In [3]:
# 2. Paths
# CHANGE THESE THREE PATHS TO YOUR FILES

TRAIN_PATH = r"C:\\Users\\shiva\\OneDrive\\Documents\\laptop\\PS\\primevul_train_paired.jsonl"
VALID_PATH = r"C:\\Users\\shiva\\OneDrive\\Documents\\laptop\\PS\\push_to_git\\primevul_valid_paired.jsonl"
TEST_PATH  = r"C:\\Users\\shiva\\OneDrive\\Documents\\laptop\\PS\\push_to_git\\primevul_test_paired.jsonl"

def load_jsonl(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line_no, line in enumerate(f, 1):
            line = line.strip()
            if not line:
                continue
            try:
                rows.append(json.loads(line))
            except json.JSONDecodeError as e:
                print(f"Bad JSON at {path}, line {line_no}: {e}")
    return pd.DataFrame(rows)

train_df = load_jsonl(TRAIN_PATH)
valid_df = load_jsonl(VALID_PATH)
test_df  = load_jsonl(TEST_PATH)

print("Train:", train_df.shape)
print("Valid:", valid_df.shape)
print("Test :", test_df.shape)
print("Columns:", list(train_df.columns))


Train: (8708, 11)
Valid: (1124, 11)
Test : (1128, 11)
Columns: ['project', 'commit_id', 'target', 'func', 'cwe', 'big_vul_idx', 'idx', 'hash', 'size', 'message', 'dataset']


In [4]:
# 3. Basic preprocessing
REQUIRED = {"func", "target"}
for name, df in [("train", train_df), ("valid", valid_df), ("test", test_df)]:
    missing = REQUIRED - set(df.columns)
    if missing:
        raise ValueError(f"{name} is missing columns: {missing}")

def clean_df(df):
    out = df.copy()
    out["func"] = out["func"].fillna("").astype(str)
    out["target"] = pd.to_numeric(out["target"], errors="coerce")
    out = out.dropna(subset=["target"])
    out["target"] = out["target"].astype(int)
    out = out[out["func"].str.strip().ne("")]
    out = out.drop_duplicates(subset=["func"], keep="first").reset_index(drop=True)
    return out

train_df = clean_df(train_df)
valid_df = clean_df(valid_df)
test_df  = clean_df(test_df)

print(train_df["target"].value_counts())
print("Null functions:", train_df["func"].isna().sum(), valid_df["func"].isna().sum(), test_df["func"].isna().sum())


target
1    4354
0    4349
Name: count, dtype: int64
Null functions: 0 0 0


## 4. Pair the vulnerable and fixed functions

RASM-Vul is built around **vulnerable/fixed pairs**. The exact pairing key depends on how your downloaded PrimeVul paired files were produced.

The function below tries common identifiers (`idx`, `big_vul_idx`, `hash`) and validates that a group contains both target 0 and target 1.

If your file already contains an explicit pair identifier, set `PAIR_COLUMN` to that column.


In [5]:
# 5. Pairing utilities
PAIR_COLUMN = None  # e.g. "idx" if your file's idx identifies a vulnerable/fixed pair

def find_pair_key(df):
    if PAIR_COLUMN and PAIR_COLUMN in df.columns:
        return PAIR_COLUMN
    for col in ["pair_id", "idx", "big_vul_idx", "hash"]:
        if col in df.columns:
            tmp = df.dropna(subset=[col])
            if len(tmp) and tmp.groupby(col)["target"].nunique().max() >= 2:
                return col
    return None

pair_key = find_pair_key(train_df)
print("Detected pair key:", pair_key)

def make_pairs(df, pair_key):
    if pair_key is None:
        raise ValueError(
            "Could not safely infer vulnerable/fixed pairs. "
            "Inspect your paired JSONL and set PAIR_COLUMN explicitly."
        )
    rows = []
    for key, g in df.groupby(pair_key, dropna=True):
        if set(g["target"].astype(int)) >= {0, 1}:
            vuln = g[g["target"] == 1].iloc[0]
            fixed = g[g["target"] == 0].iloc[0]
            rows.append({
                "pair_id": str(key),
                "vuln_func": vuln["func"],
                "fixed_func": fixed["func"],
                "cwe": vuln["cwe"] if "cwe" in vuln else None,
                "project": vuln["project"] if "project" in vuln else None,
                "commit_id": vuln["commit_id"] if "commit_id" in vuln else None
            })
    return pd.DataFrame(rows)

train_pairs = make_pairs(train_df, pair_key)
print("Training pairs:", train_pairs.shape)
print(train_pairs.head(2))


Detected pair key: big_vul_idx
Training pairs: (2050, 6)
    pair_id                                          vuln_func  \
0  177739.0   long ssl_get_algorithm2(SSL *s)\n        {\n ...   
1  177740.0  static time_t asn1_time_to_time_t(ASN1_UTCTIME...   

                                          fixed_func      cwe  project  \
0   long ssl_get_algorithm2(SSL *s)\n        {\n ...  CWE-310  openssl   
1  static time_t asn1_time_to_time_t(ASN1_UTCTIME...  CWE-119      php   

                                  commit_id  
0  ca989269a2876bae79393bd54c3e72d49975fc75  
1  c1224573c773b6845e83505f717fbf820fc18415  


In [8]:
# 6. Optional baseline
# Skip the expensive baseline for now.
# RASM-Vul does NOT depend on TF-IDF + Logistic Regression.

print("TF-IDF baseline skipped.")
print("Proceeding to Phase 1: Knowledge Extraction.")

TF-IDF baseline skipped.
Proceeding to Phase 1: Knowledge Extraction.


In [9]:
# Select one vulnerable/fixed pair from the training set

sample_pair = train_pairs.iloc[0]

print("PAIR ID:", sample_pair["pair_id"])
print("CWE:", sample_pair["cwe"])
print("PROJECT:", sample_pair["project"])

print("\n========== VULNERABLE CODE ==========\n")
print(sample_pair["vuln_func"])

print("\n========== FIXED CODE ==========\n")
print(sample_pair["fixed_func"])

PAIR ID: 177739.0
CWE: CWE-310
PROJECT: openssl

========== VULNERABLE CODE ==========

 long ssl_get_algorithm2(SSL *s)
        {
        long alg2 = s->s3->tmp.new_cipher->algorithm2;
       if (TLS1_get_version(s) >= TLS1_2_VERSION &&
            alg2 == (SSL_HANDSHAKE_MAC_DEFAULT|TLS1_PRF))
                return SSL_HANDSHAKE_MAC_SHA256 | TLS1_PRF_SHA256;
        return alg2;
	}


========== FIXED CODE ==========

 long ssl_get_algorithm2(SSL *s)
        {
        long alg2 = s->s3->tmp.new_cipher->algorithm2;
       if (s->method->version == TLS1_2_VERSION &&
            alg2 == (SSL_HANDSHAKE_MAC_DEFAULT|TLS1_PRF))
                return SSL_HANDSHAKE_MAC_SHA256 | TLS1_PRF_SHA256;
        return alg2;
	}



In [12]:
# 7. C/C++ AST using Tree-sitter
from tree_sitter import Language, Parser
import tree_sitter_cpp

CPP_LANGUAGE = Language(tree_sitter_cpp.language())
parser = Parser(CPP_LANGUAGE)

def parse_cpp(code):
    return parser.parse(code.encode("utf-8"))

def ast_type_sequence(tree_or_node):
    # If a Tree object is passed, get its root node
    if hasattr(tree_or_node, "root_node"):
        node = tree_or_node.root_node
    else:
        node = tree_or_node

    result = [node.type]

    for child in node.children:
        result.extend(ast_type_sequence(child))

    return result

def ast_structure_string(code):
    tree = parse_cpp(code)
    return " ".join(ast_type_sequence(tree.root_node))

def ast_has_error(code):
    tree = parse_cpp(code)
    return tree.root_node.has_error

print(ast_structure_string(train_df["func"].iloc[0])[:500])


translation_unit function_definition sized_type_specifier long function_declarator identifier parameter_list ( parameter_declaration type_identifier pointer_declarator * identifier ) compound_statement { declaration sized_type_specifier long init_declarator identifier = field_expression field_expression field_expression field_expression identifier -> field_identifier -> field_identifier . field_identifier -> field_identifier ; if_statement if condition_clause ( binary_expression binary_expressio


## 8. Five views

The paper describes five retrieval channels:

1. code
2. AST
3. security knowledge
4. line changes
5. AST changes

For the change views, we compare the vulnerable and fixed members of each training pair. The test function has no diff, so at inference time the query's change representation is inferred from the retrieved/predicted suspicious structure rather than from a true test diff.


In [13]:
# 9. Change-view utilities
import difflib

def line_change_text(vuln, fixed):
    a = vuln.splitlines()
    b = fixed.splitlines()
    diff = difflib.unified_diff(a, b, lineterm="", n=1)
    return "\n".join(diff)

def ast_change_text(vuln, fixed):
    a = ast_type_sequence(parse_cpp(vuln))
    b = ast_type_sequence(parse_cpp(fixed))
    sm = difflib.SequenceMatcher(a=a, b=b)
    chunks = []
    for tag, i1, i2, j1, j2 in sm.get_opcodes():
        if tag != "equal":
            chunks.append(f"{tag}: {a[i1:i2]} -> {b[j1:j2]}")
    return " | ".join(chunks)

train_pairs["line_changes"] = train_pairs.apply(
    lambda r: line_change_text(r["vuln_func"], r["fixed_func"]), axis=1
)
train_pairs["ast_changes"] = train_pairs.apply(
    lambda r: ast_change_text(r["vuln_func"], r["fixed_func"]), axis=1
)

print(train_pairs[["pair_id", "line_changes", "ast_changes"]].head(1).to_string(index=False))


 pair_id                                                                                                                                                                                                                                                line_changes                                                                                                                                                                                       ast_changes
177739.0 --- \n+++ \n@@ -3,3 +3,3 @@\n         long alg2 = s->s3->tmp.new_cipher->algorithm2;\n-       if (TLS1_get_version(s) >= TLS1_2_VERSION &&\n+       if (s->method->version == TLS1_2_VERSION &&\n             alg2 == (SSL_HANDSHAKE_MAC_DEFAULT|TLS1_PRF)) replace: ['call_expression'] -> ['field_expression', 'field_expression'] | replace: ['argument_list', '(', 'identifier', ')', '>='] -> ['->', 'field_identifier', '->', 'field_identifier', '==']


## 10. Security knowledge

The paper uses DeepSeek-V2.5 Coder to produce a nine-field structured knowledge record.

A real implementation should call the chosen LLM here. To keep this notebook runnable without an LLM API, the default uses a small **heuristic fallback**. Replace `extract_knowledge_llm()` with your DeepSeek call when you have access to the model.


In [14]:
# 11. Knowledge schema + safe fallback
KNOWLEDGE_FIELDS = [
    "context_semantics",
    "vulnerability_mechanics",
    "root_cause",
    "security_impact",
    "trigger_conditions",
    "affected_data_or_resource",
    "detection_signals",
    "remediation",
    "cwe_mapping",
]

def extract_knowledge_fallback(vuln, fixed, cwe=None):
    text = vuln.lower()
    signals = []
    if "memcpy" in text or "strcpy" in text or "strcat" in text:
        signals.append("unsafe memory/string operation")
    if "malloc" in text or "free" in text:
        signals.append("manual memory management")
    if "->" in vuln or "*" in vuln:
        signals.append("pointer/dereference operation")
    if "sql" in text:
        signals.append("SQL-related operation")
    if "if" in text or "for" in text or "while" in text:
        signals.append("control-flow validation")

    return {
        "context_semantics": "C/C++ function from a vulnerable/fixed historical pair.",
        "vulnerability_mechanics": "Potential security-relevant behavior inferred from the vulnerable function and its repair.",
        "root_cause": "Difference between the vulnerable and fixed implementations.",
        "security_impact": "Depends on the weakness represented by the pair.",
        "trigger_conditions": "Conditions under which the vulnerable operation can be reached.",
        "affected_data_or_resource": "Data, memory, pointer, or resource handled by the function.",
        "detection_signals": "; ".join(signals) if signals else "Compare vulnerable/fixed implementation changes.",
        "remediation": "Use the fixed implementation's validation or repair logic.",
        "cwe_mapping": str(cwe) if pd.notna(cwe) else "unknown",
    }

# Example:
knowledge_example = extract_knowledge_fallback(
    train_pairs.iloc[0]["vuln_func"],
    train_pairs.iloc[0]["fixed_func"],
    train_pairs.iloc[0]["cwe"]
)
knowledge_example


{'context_semantics': 'C/C++ function from a vulnerable/fixed historical pair.',
 'vulnerability_mechanics': 'Potential security-relevant behavior inferred from the vulnerable function and its repair.',
 'root_cause': 'Difference between the vulnerable and fixed implementations.',
 'security_impact': 'Depends on the weakness represented by the pair.',
 'trigger_conditions': 'Conditions under which the vulnerable operation can be reached.',
 'affected_data_or_resource': 'Data, memory, pointer, or resource handled by the function.',
 'detection_signals': 'pointer/dereference operation; control-flow validation',
 'remediation': "Use the fixed implementation's validation or repair logic.",
 'cwe_mapping': 'CWE-310'}

### Optional real LLM knowledge extraction

The following prompt is the contract your LLM call should implement. It asks for exactly the nine fields. Do not use validation/test functions to build this knowledge base.


In [17]:
# 11B. Knowledge-extraction prompt + parser

KNOWLEDGE_PROMPT = """
You are analyzing a C/C++ vulnerable/fixed pair for a vulnerability-retrieval system.

The goal is to extract GENERALIZABLE security knowledge from the repair.
Do not merely describe that the two functions are different.
Explain what security-relevant behavior is wrong, what triggers it,
why the fixed version removes or reduces the problem, and what signal
could help detect a similar vulnerability in another function.

Return ONLY a JSON object with exactly these nine keys:
context_semantics,
vulnerability_mechanics,
root_cause,
security_impact,
trigger_conditions,
affected_data_or_resource,
detection_signals,
remediation,
cwe_mapping

VULNERABLE FUNCTION:
{vuln}

FIXED FUNCTION:
{fixed}

KNOWN CWE:
{cwe}
"""

def extract_knowledge_llm(vuln, fixed, cwe, llm_generate):
    prompt = KNOWLEDGE_PROMPT.format(
        vuln=vuln,
        fixed=fixed,
        cwe=cwe
    )

    raw = llm_generate(prompt)

    try:
        obj = json.loads(raw)
    except json.JSONDecodeError as e:
        raise ValueError(
            f"DeepSeek did not return valid JSON: {e}\nRaw output:\n{raw}"
        )

    missing = set(KNOWLEDGE_FIELDS) - set(obj)
    if missing:
        raise ValueError(f"LLM omitted fields: {missing}")

    return {field: obj[field] for field in KNOWLEDGE_FIELDS}

print("Knowledge extraction prompt ready.")


Knowledge extraction prompt ready.


In [ ]:
# 11A. DeepSeek API connection

# Install once if needed:
# !pip install openai

import os
from openai import OpenAI

# IMPORTANT:
# Do not share your API key.

# Windows PowerShell:
#   $env:DEEPSEEK_API_KEY="your_key_here"

# Windows CMD:
#   set DEEPSEEK_API_KEY=your_key_here

# Temporary notebook-only option:
#   os.environ["DEEPSEEK_API_KEY"] = "your_key_here"

DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")

if not DEEPSEEK_API_KEY:
    raise ValueError(
        "DEEPSEEK_API_KEY is not set. "
        "Set it as an environment variable, then rerun this cell."
    )

client = OpenAI(
    api_key=DEEPSEEK_API_KEY,
    base_url="https://api.deepseek.com"
)

# Current DeepSeek API model name.
DEEPSEEK_MODEL = "deepseek-flash"

def llm_generate(prompt):
    response = client.chat.completions.create(
        model=DEEPSEEK_MODEL,
        messages=[
            {
                "role": "system",
                "content": (
                    "You are an expert C/C++ software vulnerability analyst. "
                    "When JSON is requested, return only valid JSON."
                )
            },
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0.3,
        response_format={"type": "json_object"}
    )

    content = response.choices[0].message.content

    if not content:
        raise ValueError("DeepSeek returned empty content.")

    return content

print("DeepSeek API connection ready:", DEEPSEEK_MODEL)


ValueError: DEEPSEEK_API_KEY is not set. Set it as an environment variable, then rerun this cell.

In [19]:
# 11C. Test knowledge extraction on ONE training pair

# Run this only after the prompt cell and DeepSeek API cell.

row = train_pairs.iloc[0]

knowledge = extract_knowledge_llm(
    row["vuln_func"],
    row["fixed_func"],
    row["cwe"],
    llm_generate
)

print(json.dumps(knowledge, indent=2, ensure_ascii=False))


NameError: name 'llm_generate' is not defined

In [ ]:
# 12. Build the Phase-1 knowledge records

def knowledge_to_text(k):
    return " | ".join(
        f"{key}: {k.get(key, '')}"
        for key in KNOWLEDGE_FIELDS
    )

# SAFETY:
# Start with one pair to verify the pipeline.
# Change KNOWLEDGE_MAX_PAIRS = None only after the one-pair test works.
USE_LLM_KNOWLEDGE = True
KNOWLEDGE_MAX_PAIRS = 1

def build_knowledge_records(train_pairs, use_llm=True, max_pairs=1):
    records = []
    n = len(train_pairs) if max_pairs is None else min(max_pairs, len(train_pairs))

    for i, (_, row) in enumerate(train_pairs.iloc[:n].iterrows(), start=1):
        if use_llm:
            k = extract_knowledge_llm(
                row["vuln_func"],
                row["fixed_func"],
                row["cwe"],
                llm_generate
            )
        else:
            k = extract_knowledge_fallback(
                row["vuln_func"],
                row["fixed_func"],
                row["cwe"]
            )

        records.append(k)
        print(f"Knowledge extracted: {i}/{n}")

    return records

knowledge_records = build_knowledge_records(
    train_pairs,
    use_llm=USE_LLM_KNOWLEDGE,
    max_pairs=KNOWLEDGE_MAX_PAIRS
)

print("\nFirst knowledge record:")
print(json.dumps(knowledge_records[0], indent=2, ensure_ascii=False))


In [ ]:
# 12A. Attach extracted knowledge to train_pairs

# During the first test, only the processed rows are populated.
train_pairs["knowledge"] = None
train_pairs["knowledge_text"] = ""

for i, k in enumerate(knowledge_records):
    idx = train_pairs.index[i]
    train_pairs.at[idx, "knowledge"] = k
    train_pairs.at[idx, "knowledge_text"] = knowledge_to_text(k)

print(
    train_pairs.loc[
        train_pairs.index[:len(knowledge_records)],
        ["pair_id", "cwe", "knowledge_text"]
    ].to_string(index=False)
)


## 13. Generate semantics-preserving variants

The poster says the training pairs are expanded using 16 semantics-preserving mutations and knowledge-guided rewrites.

Below are lightweight source-level mutations. They are intentionally conservative: a mutation is kept only if it still parses as C++.

For a strict paper reproduction, implement the exact 16 mutations used by the authors; the poster itself does not enumerate them.


In [ ]:
# 14. Conservative variant generator
def valid_cpp(code):
    try:
        return not ast_has_error(code)
    except Exception:
        return False

def make_variants(code, max_variants=16):
    candidates = []

    # Whitespace variants
    candidates.append(re.sub(r"[ \t]+", " ", code))
    candidates.append(re.sub(r"\n\s*\n+", "\n", code))

    # Comment removal
    candidates.append(re.sub(r"//.*", "", code))
    candidates.append(re.sub(r"/\*.*?\*/", "", code, flags=re.S))

    # Safe formatting-only variants
    candidates.append(code.replace("{", "{\n").replace("}", "\n}"))
    candidates.append(re.sub(r";\s*", ";\n", code))

    # Numeric spelling changes that preserve common integer meaning
    candidates.append(re.sub(r"\b0\b", "0x0", code))
    candidates.append(re.sub(r"\b1\b", "01", code))

    # Deduplicate + parse-check
    out = []
    seen = set()
    for v in candidates:
        v = v.strip()
        if v and v != code.strip() and v not in seen and valid_cpp(v):
            seen.add(v)
            out.append(v)
        if len(out) >= max_variants:
            break
    return out

variants = make_variants(train_pairs.iloc[0]["vuln_func"])
print("Variants:", len(variants))


## 15. Embeddings

The paper uses UniXCoder and 768-dimensional vectors for the five indices.

The code below loads UniXCoder through Hugging Face. If your machine cannot hold the model, use the TF-IDF fallback temporarily, but do not call that an exact reproduction.


In [ ]:
# 16. UniXCoder encoder
# Run this cell only after installing transformers + torch.
# If memory is limited, set USE_UNIXCODER = False.

USE_UNIXCODER = True
UNIXCODER_NAME = "microsoft/unixcoder-base"

if USE_UNIXCODER:
    import torch
    from transformers import RobertaTokenizer, RobertaModel

    device = "cuda" if torch.cuda.is_available() else "cpu"
    tokenizer = RobertaTokenizer.from_pretrained(UNIXCODER_NAME)
    encoder = RobertaModel.from_pretrained(UNIXCODER_NAME).to(device)
    encoder.eval()

    def unixcoder_encode(texts, batch_size=16, max_length=512):
        vectors = []
        for start in range(0, len(texts), batch_size):
            batch = texts[start:start+batch_size]
            toks = tokenizer(
                batch,
                padding=True,
                truncation=True,
                max_length=max_length,
                return_tensors="pt"
            ).to(device)
            with torch.no_grad():
                out = encoder(**toks).last_hidden_state
                mask = toks["attention_mask"].unsqueeze(-1)
                pooled = (out * mask).sum(1) / mask.sum(1).clamp(min=1)
            vectors.append(pooled.cpu().numpy())
        return np.vstack(vectors)

    print("UniXCoder ready on", device)
else:
    def unixcoder_encode(texts, batch_size=64, max_length=512):
        # Fallback, NOT paper-exact.
        vec = TfidfVectorizer(analyzer="word", max_features=768)
        return vec.fit_transform(texts).toarray().astype(np.float32)


## 16. Build the five training views

For each vulnerable/fixed pair we create:
- code view
- AST view
- knowledge view
- line-change view
- AST-change view

For retrieval, the historical case is represented by the vulnerable side. The fixed side and change information remain associated with the case.


In [ ]:
# 17. Build five-view text tables
train_pairs["code_view"] = train_pairs["vuln_func"]
train_pairs["ast_view"] = train_pairs["vuln_func"].map(ast_structure_string)
train_pairs["knowledge_view"] = train_pairs["knowledge_text"]

print(train_pairs[["code_view", "ast_view", "knowledge_view"]].head(1).T)


In [ ]:
# 18. Embed the five views
VIEW_NAMES = ["code", "ast", "knowledge", "line_changes", "ast_changes"]

view_texts = {
    "code": train_pairs["code_view"].tolist(),
    "ast": train_pairs["ast_view"].tolist(),
    "knowledge": train_pairs["knowledge_view"].tolist(),
    "line_changes": train_pairs["line_changes"].tolist(),
    "ast_changes": train_pairs["ast_changes"].tolist(),
}

embeddings = {}
for name, texts in view_texts.items():
    embeddings[name] = unixcoder_encode(texts)
    print(name, embeddings[name].shape)


## 17. Multi-view retrieval + WRRF

Each view independently ranks historical pairs.

Then WRRF combines the rankings. The weights can depend on the predicted bug type.

This is the key replacement for your current "top-5 TF-IDF majority vote".


In [ ]:
# 19. Retrieval helpers
def normalize(v):
    v = np.asarray(v, dtype=np.float32)
    n = np.linalg.norm(v)
    return v / (n + 1e-12)

def rank_view(query_vec, matrix):
    q = normalize(query_vec)
    M = matrix / (np.linalg.norm(matrix, axis=1, keepdims=True) + 1e-12)
    scores = M @ q
    order = np.argsort(-scores)
    return order, scores

def weighted_rrf(rankings, weights, k=60, top_n=10):
    fused = {}
    for view, order in rankings.items():
        w = weights.get(view, 1.0)
        for rank, idx in enumerate(order, start=1):
            fused[idx] = fused.get(idx, 0.0) + w / (k + rank)
    return sorted(fused.items(), key=lambda x: x[1], reverse=True)[:top_n]

# Simple bug-type predictor for prototype.
# Replace with the paper's learned/predicted statement-vs-structural classifier.
def predict_bug_type(code, ast_text):
    structural_tokens = [
        "class ", "template", "virtual ", "inherit", "::",
        "new ", "delete ", "goto", "switch", "try", "catch"
    ]
    if any(t in code for t in structural_tokens):
        return "structural"
    return "statement"

BUG_WEIGHTS = {
    "statement": {
        "code": 1.5, "ast": 1.0, "knowledge": 1.4,
        "line_changes": 1.2, "ast_changes": 0.8
    },
    "structural": {
        "code": 0.8, "ast": 1.6, "knowledge": 1.3,
        "line_changes": 0.9, "ast_changes": 1.5
    }
}


In [ ]:
# 20. Query one new function through all five views
def make_query_views(code, retrieved_pair=None):
    ast_view = ast_structure_string(code)

    # Query has no true diff. We use an empty/weak change view initially.
    # Later, the predicted suspicious structure can be used to improve these views.
    if retrieved_pair is None:
        line_view = ""
        ast_change_view = ""
    else:
        line_view = retrieved_pair["line_changes"]
        ast_change_view = retrieved_pair["ast_changes"]

    # Security knowledge for a raw query is initially inferred heuristically.
    qk = extract_knowledge_fallback(code, code, None)
    knowledge_view = knowledge_to_text(qk)

    return {
        "code": code,
        "ast": ast_view,
        "knowledge": knowledge_view,
        "line_changes": line_view,
        "ast_changes": ast_change_view,
    }

def retrieve(code, top_n=10):
    qviews = make_query_views(code)
    bug_type = predict_bug_type(code, qviews["ast"])

    rankings = {}
    raw_scores = {}
    for view in VIEW_NAMES:
        qvec = unixcoder_encode([qviews[view]])[0]
        order, scores = rank_view(qvec, embeddings[view])
        rankings[view] = order
        raw_scores[view] = scores

    fused = weighted_rrf(rankings, BUG_WEIGHTS[bug_type], top_n=top_n)
    return qviews, bug_type, fused, raw_scores

# Test with one validation function
example_code = valid_df["func"].iloc[0]
qviews, bug_type, fused, raw_scores = retrieve(example_code, top_n=5)

print("Predicted bug type:", bug_type)
print("Top retrieved pair indices:", fused)


## 18. LLM re-ranking

RASM-Vul then asks an LLM to re-rank the retrieved cases by relevance.

The adapter below accepts any function that takes a prompt and returns text. You can connect DeepSeek, another local model, or an API later.


In [ ]:
# 21. Re-ranking prompt + deterministic fallback
RERANK_PROMPT = '''
Rank these historical vulnerability cases for the NEW FUNCTION.
Return JSON only:
{
  "ranked_pair_ids": ["..."],
  "reason": "short reason"
}

NEW FUNCTION:
{query}

HISTORICAL CASES:
{cases}
'''

def rerank_fallback(query, candidates):
    # Retrieval score is used as the fallback ranking.
    ranked = sorted(candidates, key=lambda x: x["rrf_score"], reverse=True)
    return ranked, "Fallback ranking by WRRF score."

def llm_rerank(query, candidates, llm_generate=None):
    if llm_generate is None:
        return rerank_fallback(query, candidates)

    cases_text = []
    for c in candidates:
        cases_text.append(
            f"PAIR_ID={c['pair_id']}\n"
            f"VULNERABLE=\n{c['vuln_func']}\n"
            f"FIXED=\n{c['fixed_func']}\n"
            f"KNOWLEDGE={c['knowledge_text']}\n"
            f"WRRF={c['rrf_score']}"
        )

    prompt = RERANK_PROMPT.format(query=query, cases="\n\n---\n\n".join(cases_text))
    result = json.loads(llm_generate(prompt))
    order = {pid: i for i, pid in enumerate(result["ranked_pair_ids"])}
    ranked = sorted(candidates, key=lambda x: order.get(x["pair_id"], 999))
    return ranked, result.get("reason", "")


## 19. Final decision

The final LLM sees the new function plus the strongest historical matches and returns:

- `is_vul`
- `cwe`
- `rationale`
- `signal`

This mirrors the poster's output contract.


In [ ]:
# 22. Final decision prompt + fallback
DECISION_PROMPT = '''
You are a C/C++ vulnerability analyst.

Given a NEW FUNCTION and historical vulnerable/fixed examples, decide whether
the NEW FUNCTION is vulnerable.

Return ONLY JSON:
{
  "is_vul": true,
  "cwe": "CWE-XXX or unknown",
  "rationale": "short explanation",
  "signal": "strong|medium|weak"
}

NEW FUNCTION:
{query}

HISTORICAL EVIDENCE:
{evidence}
'''

def decision_fallback(query, ranked_cases):
    if not ranked_cases:
        return {
            "is_vul": False,
            "cwe": "unknown",
            "rationale": "No historical evidence was retrieved.",
            "signal": "weak"
        }

    best = ranked_cases[0]
    # Prototype only: nearest historical target is the decision.
    # Historical pair itself is not a direct truth label for an arbitrary query.
    # We therefore use a conservative evidence-based signal.
    return {
        "is_vul": True,
        "cwe": str(best.get("cwe", "unknown")) if pd.notna(best.get("cwe")) else "unknown",
        "rationale": "A highly ranked historical vulnerable/fixed case was retrieved; human review is required.",
        "signal": "strong" if best["rrf_score"] > 0.03 else "medium"
    }

def final_decision(query, ranked_cases, llm_generate=None):
    if llm_generate is None:
        return decision_fallback(query, ranked_cases)

    evidence = "\n\n---\n\n".join(
        f"PAIR={c['pair_id']}\nVULNERABLE={c['vuln_func']}\nFIXED={c['fixed_func']}\nKNOWLEDGE={c['knowledge_text']}"
        for c in ranked_cases[:5]
    )
    prompt = DECISION_PROMPT.format(query=query, evidence=evidence)
    return json.loads(llm_generate(prompt))


In [ ]:
# 23. Complete end-to-end function
def analyze_function(code, llm_generate=None, top_n=5):
    qviews, bug_type, fused, raw_scores = retrieve(code, top_n=top_n)

    candidates = []
    for idx, score in fused:
        row = train_pairs.iloc[idx]
        candidates.append({
            "pair_id": str(row["pair_id"]),
            "vuln_func": row["vuln_func"],
            "fixed_func": row["fixed_func"],
            "knowledge_text": row["knowledge_text"],
            "cwe": row["cwe"],
            "rrf_score": float(score),
            "project": row["project"],
            "commit_id": row["commit_id"],
        })

    ranked, rerank_reason = llm_rerank(code, candidates, llm_generate=llm_generate)
    decision = final_decision(code, ranked, llm_generate=llm_generate)

    result = {
        "is_vul": bool(decision["is_vul"]),
        "cwe": decision.get("cwe", "unknown"),
        "rationale": decision.get("rationale", ""),
        "signal": decision.get("signal", "weak"),
        "bug_type": bug_type,
        "retrieved_cases": [
            {
                "pair_id": c["pair_id"],
                "rrf_score": c["rrf_score"],
                "project": c["project"],
                "cwe": c["cwe"],
            }
            for c in ranked
        ],
        "rerank_reason": rerank_reason,
    }
    return result

result = analyze_function(example_code)
print(json.dumps(result, indent=2, default=str))


## 20. Evaluation without leakage

For evaluation:
- Build all retrieval indexes using **training pairs only**.
- Use validation/test functions only as queries.
- Never insert validation/test functions into the training index.
- Compare the final predictions with the known labels.

The paper reports accuracy, F1, recall and paired accuracy. The poster also emphasizes that this is a recall-first screening system and that human review is still required.


In [ ]:
# 24. Simple retrieval evaluation on a small sample first
def evaluate_sample(df, n=20):
    rows = []
    sample = df.head(min(n, len(df)))
    for _, r in sample.iterrows():
        pred = analyze_function(r["func"])
        rows.append({
            "true": int(r["target"]),
            "pred": int(pred["is_vul"]),
            "signal": pred["signal"]
        })
    out = pd.DataFrame(rows)
    print(classification_report(out["true"], out["pred"], digits=4, zero_division=0))
    return out

# Start small because UniXCoder inference is expensive.
# eval_df = evaluate_sample(valid_df, n=20)


# Final architecture implemented

```text
PrimeVul paired training data
        ↓
Preprocessing + pair validation
        ↓
Vulnerable / fixed pairs
        ↓
Knowledge extraction
        ↓
Variants
        ↓
Five views
 ┌──────┬─────┬──────────┬──────────────┬────────────┐
 │ code │ AST │ knowledge│ line changes │ AST changes│
 └──────┴─────┴──────────┴──────────────┴────────────┘
        ↓
UniXCoder embeddings
        ↓
Five retrieval indexes
        ↓
New C/C++ function
        ↓
Bug-type prediction
        ↓
Multi-view retrieval
        ↓
WRRF
        ↓
LLM re-ranking
        ↓
LLM decision
        ↓
JSON:
is_vul + CWE + rationale + signal
```

**Important:** the heuristic knowledge extractor, conservative mutations, simple bug-type predictor, and fallback decision are prototype substitutes. For a faithful reproduction, replace those parts with the corresponding paper-specific LLM/mutation/prediction implementations.
